In [6]:
import polars as pl


DATASET_NAME_PERFIX = "dataset_255w_"
TRAIN_SIZE = 0.8
VAL_SIZE = 0.1
TEST_SIZE = 0.1


df_with_parent= pl.read_csv(
    "lehner_dataset_with_sequences.csv",
    separator=",",
    null_values="NA",
    infer_schema_length=10000
)
df_with_parent = df_with_parent.drop_nulls(subset=["original_sequence", "position", "normalized_fitness", "normalized_fitness_sigma"])

df_with_parent = df_with_parent.drop(["domain_ID", "aa_seq", "input_count_rep1", "input_count_rep2", "input_count_rep3",
                 "output_count_rep1", "output_count_rep2", "output_count_rep3",
                 "mean_input_count", "quality_rank", "fitness", 'fitness_sigma'])

df_with_parent

uniprot_ID,wt_aa,position,mut_aa,STOP,normalized_fitness,normalized_fitness_sigma,original_sequence,normalized_fitness_training,normalized_fitness_training_sigmoid,normalized_fitness_training_tanh,normalized_fitness_asymmetric
str,str,i64,str,bool,f64,f64,str,f64,f64,f64,f64
"""A0A2R8Y422""","""Q""",2,"""*""",true,-0.81905,0.208478,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",-0.081722,-0.081722,-0.102025,-0.530676
"""A0A2R8Y422""","""Q""",2,"""A""",false,-0.28079,0.093461,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",-0.028072,-0.028072,-0.035084,-0.510528
"""A0A2R8Y422""","""Q""",2,"""C""",false,-0.10325,0.057995,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",-0.010325,-0.010325,-0.012906,-0.503872
"""A0A2R8Y422""","""Q""",2,"""D""",false,-0.258003,0.072296,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",-0.025795,-0.025795,-0.032239,-0.509674
"""A0A2R8Y422""","""Q""",2,"""E""",false,0.376783,0.169263,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",0.03766,0.03766,0.047063,0.528229
…,…,…,…,…,…,…,…,…,…,…,…
"""Q9Y6V0""","""C""",1059,"""V""",false,-1.113722,0.157535,"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…",-0.110914,-0.110914,-0.138323,-0.541668
"""Q9Y6V0""","""C""",1059,"""W""",false,-0.903822,0.15048,"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…",-0.090137,-0.090137,-0.112499,-0.533841
"""Q9Y6V0""","""C""",1059,"""Y""",false,-1.149495,0.155403,"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…",-0.114446,-0.114446,-0.142706,-0.543


In [2]:
df_with_parent["normalized_fitness_training"].describe()

statistic,value
str,f64
"""count""",560668.0
"""null_count""",0.0
"""mean""",-0.031822
"""std""",0.038346
"""min""",-0.485193
"""25%""",-0.057357
"""50%""",-0.021016
"""75%""",-0.003941
"""max""",0.202422


In [3]:



# Remove rows with NaN in original_sequence and position
df_127 = df_with_parent.drop_nulls(subset=["original_sequence", "position", "normalized_fitness", "normalized_fitness_sigma"])


# Add sub_sequence column (21 aa from 'position')
df_127 = df_127.with_columns(
    pl.col("original_sequence").str.slice(pl.col("position").cast(pl.Int64), 21).alias("sub_sequence")
)

WINDOW = 255
HALF = WINDOW // 2  # 256

def extract_fragment(seq: str, pos: int, mut_aa=None) -> tuple[str, int]:
    if seq is None or pos is None:
        return None, None

    pos -= 1  # Convert to 0-based index
    seq_len = len(seq)
    # Navrhujeme start/end tak, aby pos byl uprostřed
    start = max(0, pos - HALF)
    end = min(seq_len, start + WINDOW)

    # Pokud na konci nezbyde 512 znaků, posuň začátek zpět
    if end - start < WINDOW:
        start = max(0, end - WINDOW)

    if mut_aa:
        seqlist = list(seq)
        seqlist[pos] = mut_aa
        seq = "".join(seqlist)

    fragment = seq[start:end]
    local_pos = pos - start
    return fragment, local_pos

def position_normilized(seq: str, pos: int) -> int:
    if seq is None or pos is None:
        return None
    pos -= 1  # Convert to 0-based index
    seq_len = len(seq)
    if pos < 0 or pos >= seq_len:
        return None
    return pos/ seq_len

df_127 = df_127.with_columns([
    pl.struct(["original_sequence", "position", "mut_aa"])
      .map_elements(lambda x: extract_fragment(x["original_sequence"], int(x["position"]), x["mut_aa"])[0],
                    return_dtype=pl.Utf8)
      .alias("fragment_255_mut"),

    pl.struct(["original_sequence", "position"])
      .map_elements(lambda x: extract_fragment(x["original_sequence"], int(x["position"]))[1],
                    return_dtype=pl.Int64)
      .alias("local_position"),

    pl.struct(["original_sequence", "position"])
      .map_elements(lambda x: extract_fragment(x["original_sequence"], int(x["position"]))[0],
                    return_dtype=pl.Utf8)
      .alias("fragment_255_org"),

        pl.struct(["original_sequence", "position"])
      .map_elements(lambda x: position_normilized(x["original_sequence"], int(x["position"])),
                    return_dtype=pl.Float32)
      .alias("position_normilized"),
])


df_127 = df_127.drop(["original_sequence", "sub_sequence", "position"])

df_127


uniprot_ID,wt_aa,mut_aa,STOP,normalized_fitness,normalized_fitness_sigma,normalized_fitness_training,normalized_fitness_training_sigmoid,normalized_fitness_training_tanh,normalized_fitness_asymmetric,fragment_255_mut,local_position,fragment_255_org,position_normilized
str,str,str,bool,f64,f64,f64,f64,f64,f64,str,i64,str,f32
"""A0A2R8Y422""","""Q""","""*""",true,-0.81905,0.208478,-0.081722,-0.081722,-0.102025,-0.530676,"""M*IFVKTLMGKTITLEVELSDTIDNVKAKI…",1,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",0.00641
"""A0A2R8Y422""","""Q""","""A""",false,-0.28079,0.093461,-0.028072,-0.028072,-0.035084,-0.510528,"""MAIFVKTLMGKTITLEVELSDTIDNVKAKI…",1,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",0.00641
"""A0A2R8Y422""","""Q""","""C""",false,-0.10325,0.057995,-0.010325,-0.010325,-0.012906,-0.503872,"""MCIFVKTLMGKTITLEVELSDTIDNVKAKI…",1,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",0.00641
"""A0A2R8Y422""","""Q""","""D""",false,-0.258003,0.072296,-0.025795,-0.025795,-0.032239,-0.509674,"""MDIFVKTLMGKTITLEVELSDTIDNVKAKI…",1,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",0.00641
"""A0A2R8Y422""","""Q""","""E""",false,0.376783,0.169263,0.03766,0.03766,0.047063,0.528229,"""MEIFVKTLMGKTITLEVELSDTIDNVKAKI…",1,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",0.00641
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Q9Y6V0""","""C""","""V""",false,-1.113722,0.157535,-0.110914,-0.110914,-0.138323,-0.541668,"""FGFGASIFSQASNLISTAGQPGPHSQSGPG…",127,"""FGFGASIFSQASNLISTAGQPGPHSQSGPG…",0.205757
"""Q9Y6V0""","""C""","""W""",false,-0.903822,0.15048,-0.090137,-0.090137,-0.112499,-0.533841,"""FGFGASIFSQASNLISTAGQPGPHSQSGPG…",127,"""FGFGASIFSQASNLISTAGQPGPHSQSGPG…",0.205757
"""Q9Y6V0""","""C""","""Y""",false,-1.149495,0.155403,-0.114446,-0.114446,-0.142706,-0.543,"""FGFGASIFSQASNLISTAGQPGPHSQSGPG…",127,"""FGFGASIFSQASNLISTAGQPGPHSQSGPG…",0.205757


In [4]:
# First, we shuffle the data randomly
df_shuffled = df_127.sample(fraction=1, shuffle=True, seed=42) # Using a seed for reproducibility

# Get the total number of rows
n_rows = len(df_shuffled)

# Calculate the sizes of each set (80% training, 10% validation, 10% testing)
train_size = int(TRAIN_SIZE * n_rows)
val_size = int(VAL_SIZE * n_rows)
# The remainder will be the testing set

# Split the dataframe into three parts
train_df = df_shuffled.slice(0, train_size)
val_df = df_shuffled.slice(train_size, val_size)
test_df = df_shuffled.slice(train_size + val_size) # to the end


# Save each dataset to a separate CSV file
train_df.write_csv(f"{DATASET_NAME_PERFIX}train.csv")
val_df.write_csv(f"{DATASET_NAME_PERFIX}validation.csv")
test_df.write_csv(f"{DATASET_NAME_PERFIX}test.csv")

print(f"Datasets were successfully created and saved:")
print(f"- Training set (training_dataset.csv): {len(train_df)} rows")
print(f"- Validation set (validation_dataset.csv): {len(val_df)} rows")
print(f"- Testing set (testing_dataset.csv): {len(test_df)} rows")



Datasets were successfully created and saved:
- Training set (training_dataset.csv): 448534 rows
- Validation set (validation_dataset.csv): 56066 rows
- Testing set (testing_dataset.csv): 56068 rows


In [5]:
train_df

uniprot_ID,wt_aa,mut_aa,STOP,normalized_fitness,normalized_fitness_sigma,normalized_fitness_training,normalized_fitness_training_sigmoid,normalized_fitness_training_tanh,normalized_fitness_asymmetric,fragment_255_mut,local_position,fragment_255_org,position_normilized
str,str,str,bool,f64,f64,f64,f64,f64,f64,str,i64,str,f32
"""Q14839""","""C""","""*""",true,-0.918931,0.204821,-0.091635,-0.091635,-0.114364,-0.534405,"""KPKKVAPLKIKLGGFGSKRKRSSSEDDDLD…",127,"""KPKKVAPLKIKLGGFGSKRKRSSSEDDDLD…",0.216004
"""P13796""","""K""","""D""",false,-0.0446,0.051003,-0.00446,-0.00446,-0.005575,-0.501672,"""MARGSVSDEEMMELREAFAKVDTDGNGYIS…",81,"""MARGSVSDEEMMELREAFAKVDTDGNGYIS…",0.129187
"""Q13642""","""D""","""S""",false,-0.090727,0.06228,-0.009072,-0.009072,-0.01134,-0.503402,"""AKCLHPLANETFVAKDNKILCNKCTTREDS…",142,"""AKCLHPLANETFVAKDNKILCNKCTTREDS…",0.650155
"""Q9BZM3""","""W""","""C""",false,-0.740526,0.131823,-0.073918,-0.073918,-0.092302,-0.527741,"""CPSRKSGAFCVCPLCVTSHLHSSRGSVGAG…",199,"""CPSRKSGAFCVCPLCVTSHLHSSRGSVGAG…",0.815789
"""P50539""","""H""","""M""",false,-0.107661,0.105316,-0.010766,-0.010766,-0.013457,-0.504037,"""MERVKMINVQRLLEAAEFLERRERECEHGY…",102,"""MERVKMINVQRLLEAAEFLERRERECEHGY…",0.447368
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""O94885""","""I""","""P""",false,-0.994216,0.098684,-0.099095,-0.099095,-0.123641,-0.537214,"""QSSMSGQTVSTTDSSTSNRESVKSEDGDDE…",127,"""QSSMSGQTVSTTDSSTSNRESVKSEDGDDE…",0.516439
"""O43167""","""K""","""F""",false,0.018182,0.061021,0.001818,0.001818,0.002273,0.501364,"""ERPFKCNECGKGFAQKHSLQVHTRMHTGER…",127,"""ERPFKCNECGKGFAQKHSLQVHTRMHTGER…",0.638451
"""P07814""","""K""","""C""",false,-0.293656,0.096327,-0.029357,-0.029357,-0.036691,-0.51101,"""PSLNNNCTTSEDSLVLYNRVAVQGDVVREL…",127,"""PSLNNNCTTSEDSLVLYNRVAVQGDVVREL…",0.571429
